# Autoencoder Retraining Stability (Identity Activation)

**Experiment Design:**
1. Train AE multiple times with different random seeds for each bottleneck size
2. Use **Identity** activation in encoder/decoder (linear AE) instead of LeakyReLU
3. Compute pairwise correlations between all embedding pairs (after Procrustes alignment)

**Metrics:**
- Mean pairwise correlation across runs
- Per-OA standard deviation across runs
- Reconstruction error stability (RMSE mean ± std)

**Note:** With Identity activations, the AE becomes a linear model — comparable to PCA. Useful as a baseline against the nonlinear (LeakyReLU) variant.

## 1. Setup and Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import yaml
from torchgeodemo import autoencoder_train_latent
from scipy.spatial import procrustes
from scipy.stats import pearsonr
import geopandas as gpd
import os
import pickle
from tqdm import tqdm
from itertools import combinations
import json
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch version: 2.10.0+cu128
CUDA available: True


## 2. Configuration

In [ ]:
# Helper function for layer size generation (from notebook 2)
def gen_layer_sizes(input_size, latent_size, num_layers, scaling_type="lin"):
    """Generate encoder/decoder layer widths."""
    if scaling_type == "mul":
        scale = (latent_size / input_size) ** (1 / (num_layers - 1))
        return [int(input_size * scale ** i) for i in range(1, num_layers)]
    if scaling_type == "lin":
        step = (latent_size - input_size) / (num_layers - 1)
        return [int(input_size + step * i) for i in range(1, num_layers)]
    raise ValueError("Invalid scaling type. Use 'mul' or 'lin'.")

# Training parameters
batch_size = 0.01  # 1% of data
scaling_type = "lin"

# Paths — output_dir name keeps the "500epochs" tag for continuity, even though some
# bottlenecks now train for longer (100/128 had not converged at 500ep).
output_dir_epoch_tag = 500
data_path = "../data/census_data/engcensus_cleaned_scaled.parquet"
geofile_path = "../data/geofiles/Output_Areas_(December_2021)_Boundaries_EW_BFE_(V9)_and_RUC.geojson"
output_dir = f"../AE_outputs/retraining_stability_{output_dir_epoch_tag}epochs_{scaling_type}scaling_identity"
os.makedirs(output_dir, exist_ok=True)
os.makedirs(f"{output_dir}/data/", exist_ok=True)
os.makedirs(f"{output_dir}/models/", exist_ok=True)
os.makedirs(f"{output_dir}/models/yamls/", exist_ok=True)
os.makedirs(f"{output_dir}/yamls/", exist_ok=True)

# Identity AE is linear → single run per bottleneck.
bottleneck_sizes = [128, 100, 64, 32, 16, 8, 4, 2]

# Per-bottleneck epoch budget. 100/128D had not converged at 500ep or 1500ep — at 1500ep
# the loss was still dropping several % over the final 10% of training, so give them a
# much longer budget.
n_epochs_by_dim = {
    2:   500,
    4:   500,
    8:   500,
    16:  500,
    32:  500,
    64:  500,
    100: 3000,
    128: 4000,
}

# Per-bottleneck optimizer schedule. torchgeodemo defaults
# (lr=1e-3, ReduceLROnPlateau factor=0.2, patience=10, min_lr=1e-7) decay the LR
# toward min_lr long before epoch 1500 at high dim — that's why 100/128D plateau
# above PCA. For those dims, raise the LR floor and slow the decay so optimisation
# can keep making progress over a longer budget.
default_lr_cfg = dict(lr=1e-3, factor=0.2, patience=10, min_lr=1e-7)
lr_cfg_by_dim = {
    2:   default_lr_cfg,
    4:   default_lr_cfg,
    8:   default_lr_cfg,
    16:  default_lr_cfg,
    32:  default_lr_cfg,
    64:  default_lr_cfg,
    100: dict(lr=3e-3, factor=0.5, patience=30, min_lr=1e-5),
    128: dict(lr=3e-3, factor=0.5, patience=30, min_lr=1e-5),
}

n_runs = 1
base_seed = 20210321

print("Configuration:")
print(f"  Bottleneck sizes: {bottleneck_sizes}")
print(f"  Epochs per bottleneck: {n_epochs_by_dim}")
print(f"  LR schedule per bottleneck: {lr_cfg_by_dim}")
print(f"  Number of runs per size: {n_runs}")
print(f"  Total models to train: {len(bottleneck_sizes) * n_runs}")
print(f"  Base seed: {base_seed}")
print(f"  Batch size: {batch_size} ({int(batch_size*100)}% of data)")
print(f"  Layer scaling: {scaling_type}")
print(f"  Output directory: {output_dir}")

## 3. Utility Functions

In [3]:
def procrustes_align(X_source, X_target):
    """
    Align X_source to X_target using Procrustes transformation.
    Returns: aligned X_source, disparity
    """
    mtx1, mtx2, disparity = procrustes(X_target, X_source)
    return mtx1, disparity

def compute_pairwise_correlations(embeddings_list):
    """
    Compute pairwise correlations between all embedding pairs.
    
    Parameters:
    - embeddings_list: list of embedding arrays (each shape: n_samples x n_dims)
    
    Returns:
    - mean_correlation: mean across all pairs
    - correlation_matrix: n_runs x n_runs matrix of correlations
    - all_correlations: flat array of all pairwise correlations
    """
    n_runs = len(embeddings_list)
    correlation_matrix = np.ones((n_runs, n_runs))
    all_correlations = []
    
    for i, j in combinations(range(n_runs), 2):
        # Align j to i using Procrustes
        aligned_j, _ = procrustes_align(embeddings_list[j], embeddings_list[i])
        
        # Compute per-dimension correlations
        dim_corrs = []
        for dim in range(embeddings_list[i].shape[1]):
            corr, _ = pearsonr(embeddings_list[i][:, dim], aligned_j[:, dim])
            dim_corrs.append(corr)
        
        mean_corr = np.mean(dim_corrs)
        correlation_matrix[i, j] = mean_corr
        correlation_matrix[j, i] = mean_corr
        all_correlations.append(mean_corr)
    
    return np.mean(all_correlations), correlation_matrix, np.array(all_correlations)

def compute_per_oa_std(embeddings_list):
    """
    Compute per-OA standard deviation across runs.
    First aligns all embeddings to the first run using Procrustes.
    
    Returns: array of shape (n_samples,) with std for each OA
    """
    n_runs = len(embeddings_list)
    reference = embeddings_list[0]
    
    # Align all to reference
    aligned_embeddings = [reference]
    for i in range(1, n_runs):
        aligned, _ = procrustes_align(embeddings_list[i], reference)
        aligned_embeddings.append(aligned)
    
    # Stack and compute std across runs
    stacked = np.stack(aligned_embeddings, axis=0)  # shape: (n_runs, n_samples, n_dims)
    per_oa_std = np.mean(stacked.std(axis=0), axis=1)  # mean std across dimensions
    
    return per_oa_std

print("Utility functions defined")

Utility functions defined


## 4. Helper Functions for AE Training via YAML

In [4]:
def train_ae_via_yaml(data_df, run_name, latent_dim, working_dir, seed,
                      n_epochs=250, batch_size=0.01, scaling_type="lin"):
    """
    Train autoencoder using torchgeodemo with YAML configuration.
    Skips training if output files already exist.
    
    Parameters:
    - data_df: DataFrame with data (must have 'OA' column)
    - run_name: name for this run (e.g., 'run_0', 'run_1')
    - latent_dim: bottleneck dimension
    - working_dir: directory for outputs
    - seed: random seed for this run
    - n_epochs: training epochs
    - batch_size: batch size fraction
    - scaling_type: 'lin' or 'mul' for layer scaling
    
    Returns: embeddings (numpy array)
    """
    # torchgeodemo saves files with double underscores directly in working_dir
    # Include n_epochs in filename to distinguish different training runs
    model_nickname = f"stability_{run_name}__ae_{latent_dim}d_{run_name}_{n_epochs}ep_v1"
    latent_csv_path = f"{working_dir}/{model_nickname}__latent.csv"
    
    # Check if already trained - skip if latent CSV exists
    if os.path.exists(latent_csv_path):
        print(f"[{run_name}] Found existing output ({n_epochs} epochs), skipping training")
        latent_df = pd.read_csv(latent_csv_path, index_col="OA")
        embeddings = latent_df.values
        print(f"[{run_name}] Loaded embeddings: {embeddings.shape}")
        return embeddings
    
    print(f"[{run_name}] Training (seed={seed}, epochs={n_epochs})")
    torch.manual_seed(seed)
    
    if 'OA' not in data_df.columns:
        data_df = data_df.reset_index()
    
    input_dim = data_df.shape[1] - 1
    encoder_sizes = gen_layer_sizes(input_dim, latent_dim, num_layers=4, scaling_type=scaling_type)
    
    yaml_config = {
        "data": {
            "source": "TEMP",
            "nickname": f"stability_{run_name}",
            "id_col": "OA"
        },
        "working_dir": working_dir,
        "autoencoder": {
            "nickname": f"ae_{latent_dim}d_{run_name}_{n_epochs}ep",
            "version": "1",
            "save_latent": "csv",
            "max_epochs": n_epochs,
            "batch_size": batch_size,
            "use_covariance_loss": False,
            "random_seed": seed,
            "encoder": {
                "sizes": encoder_sizes,
                "activation": "Identity"
            },
            "decoder": {
                "sizes": encoder_sizes[::-1],
                "activation": "Identity"
            }
        }
    }
    
    temp_data_path = f"{working_dir}/temp_data_{run_name}.parquet"
    data_df.to_parquet(temp_data_path)
    yaml_config["data"]["source"] = temp_data_path
    
    yaml_dir = f"{working_dir}/yamls"
    os.makedirs(yaml_dir, exist_ok=True)
    config_path = f"{yaml_dir}/config_{run_name}_{latent_dim}d_{n_epochs}ep.yaml"
    with open(config_path, 'w') as f:
        yaml.dump(yaml_config, f, default_flow_style=False)
    
    autoencoder_train_latent.main(config_path, create_latent=True, save_reco_error=False, verbose=True)
    print(f"[{run_name}] Training complete")
    
    if os.path.exists(latent_csv_path):
        latent_df = pd.read_csv(latent_csv_path, index_col="OA")
        embeddings = latent_df.values
    else:
        model_path = f"{working_dir}/{model_nickname}__model.pth"
        print(f"[{run_name}] Latent CSV missing, encoding manually")
        embeddings = load_ae_and_encode(model_path, data_df.drop(columns=['OA']))
    
    print(f"[{run_name}] Embeddings shape: {embeddings.shape}")
    return embeddings

def load_ae_and_encode(model_path, X_data):
    """Load trained AE model and encode data"""
    from torchgeodemo.models import AutoEncoder
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = torch.load(model_path, map_location=device, weights_only=False)
    model.eval()
    
    if isinstance(X_data, pd.DataFrame):
        X_np = X_data.values
    else:
        X_np = X_data
    
    with torch.no_grad():
        X_tensor = torch.FloatTensor(X_np).to(device)
        embeddings = model.encode(X_tensor).cpu().numpy()
    
    return embeddings

print("YAML-based training and loading functions defined")

YAML-based training and loading functions defined


## 5. Load Data

In [5]:
# Load census data as DataFrame
df = pd.read_parquet(data_path)
print(f"Census data shape: {df.shape}")

# Ensure OA column exists
if 'OA' not in df.columns:
    df = df.reset_index()

# Create numpy version for processing
X_full = df.drop(columns=['OA']).values
oa_ids = df['OA'].values

print(f"\nData summary:")
print(f"  Total OAs: {len(df)}")
print(f"  Variables: {X_full.shape[1]}")
print(f"  Data range: [{X_full.min():.4f}, {X_full.max():.4f}]")

Census data shape: (188880, 409)

Data summary:
  Total OAs: 188880
  Variables: 408
  Data range: [0.0000, 1.0000]


# =============================================================================
# PART 1: AUTOENCODER RETRAINING STABILITY
# =============================================================================

In [6]:
# Helper function to check if a bottleneck size has been completed
def get_checkpoint_path(bottleneck_size):
    return f"{output_dir}/data/stability_checkpoint_{bottleneck_size}d.pkl"

def load_checkpoint(bottleneck_size):
    """Load checkpoint for a specific bottleneck size if it exists."""
    checkpoint_path = get_checkpoint_path(bottleneck_size)
    if os.path.exists(checkpoint_path):
        with open(checkpoint_path, 'rb') as f:
            return pickle.load(f)
    return None

def save_checkpoint(bottleneck_size, embeddings_list, reco_errors_list):
    """Save checkpoint for a specific bottleneck size."""
    checkpoint_path = get_checkpoint_path(bottleneck_size)
    checkpoint = {
        'bottleneck_size': bottleneck_size,
        'embeddings_list': embeddings_list,
        'reco_errors_list': reco_errors_list,
        'n_runs': len(embeddings_list)
    }
    with open(checkpoint_path, 'wb') as f:
        pickle.dump(checkpoint, f)
    print(f"  Checkpoint saved: {checkpoint_path}")

print("Checkpoint functions defined")

Checkpoint functions defined


In [ ]:
# --- Wipe stale identity-AE outputs ---
# Existing models in output_dir were trained BEFORE the MLP monkey-patch above, so they
# are actually nonlinear LeakyReLU AEs mislabelled as "identity". Delete them so the
# training loop retrains everything as a true linear AE.
#
# Set WIPE_STALE = True the first time you re-run; leave False afterwards.

WIPE_STALE = False

if WIPE_STALE:
    import glob
    targets = (
        glob.glob(f"{output_dir}/models/stability_*__model.pth")
        + glob.glob(f"{output_dir}/models/stability_*__latent.csv")
        + glob.glob(f"{output_dir}/models/stability_*__model__info.txt")
        + glob.glob(f"{output_dir}/data/stability_checkpoint_*d.pkl")
        + glob.glob(f"{output_dir}/data/all_stability_results.pkl")
    )
    for p in targets:
        os.remove(p)
    print(f"Removed {len(targets)} stale files from {output_dir}")
    print("Re-run the training cell below to retrain as a genuine linear AE.")
else:
    print("WIPE_STALE=False — keeping existing files (set True to force retrain).")

Removed 33 stale files from ../AE_outputs/retraining_stability_500epochs_linscaling_identity
Re-run the training cell below to retrain as a genuine linear AE.


In [8]:
# --- Monkey-patch torchgeodemo.MLP so hidden activations are Identity ---
# The library hardcodes LeakyReLU between hidden Linear layers (models.py L57); the
# `activation` YAML knob only controls the *final* layer. To get a genuinely linear AE
# (comparable to PCA), we patch MLP.__init__ so any LeakyReLU inserted during
# construction is immediately replaced with nn.Identity.
#
# Run this cell BEFORE the training loop. The patch persists for the rest of the kernel.

from torchgeodemo import models as _tgd_models
import torch.nn as _nn

if not getattr(_tgd_models.MLP, "_linearized", False):
    _orig_MLP_init = _tgd_models.MLP.__init__

    def _linear_MLP_init(self, *args, **kwargs):
        _orig_MLP_init(self, *args, **kwargs)
        # Walk the Sequential and replace LeakyReLU -> Identity
        for i, layer in enumerate(self.mlp):
            if isinstance(layer, _nn.LeakyReLU):
                self.mlp[i] = _nn.Identity()

    _tgd_models.MLP.__init__ = _linear_MLP_init
    _tgd_models.MLP._linearized = True
    print("Patched torchgeodemo.models.MLP: hidden LeakyReLU -> Identity")
else:
    print("torchgeodemo.models.MLP already patched (linearized)")

# Sanity check: build a tiny MLP and confirm no LeakyReLU remains
_m = _tgd_models.MLP([10, 8, 4])
assert not any(isinstance(l, _nn.LeakyReLU) for l in _m.mlp), "Patch failed: LeakyReLU still present"
print("Verified: no LeakyReLU in patched MLP")
print(_m.mlp)

Patched torchgeodemo.models.MLP: hidden LeakyReLU -> Identity
Verified: no LeakyReLU in patched MLP
Sequential(
  (0): Linear(in_features=10, out_features=8, bias=True)
  (1): Identity()
  (2): Linear(in_features=8, out_features=4, bias=True)
  (3): Identity()
)


In [ ]:
# --- Monkey-patch torchgeodemo AutoEncoder.configure_optimizers ---
# Library default is AdamW(lr=1e-3) + ReduceLROnPlateau(factor=0.2, patience=10,
# min_lr=1e-7). With patience=10 the LR collapses toward min_lr long before our
# 1500-epoch budget for 100/128D, leaving the linear AE stuck above PCA.
#
# Patch reads `_CURRENT_LR_CFG` (a module global set by the training loop just
# before each fit) so each bottleneck can use its own schedule.

import torch as _torch
from torchgeodemo import models as _tgd_models2

_CURRENT_LR_CFG = dict(lr=1e-3, factor=0.2, patience=10, min_lr=1e-7)

def _patched_configure_optimizers(self):
    cfg = _CURRENT_LR_CFG
    optimizer = _torch.optim.AdamW(self.parameters(), lr=cfg["lr"])
    scheduler = _torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=cfg["factor"],
        patience=cfg["patience"],
        min_lr=cfg["min_lr"],
    )
    return {"optimizer": optimizer, "lr_scheduler": scheduler, "monitor": "train_loss_epoch"}

# Find the AE class (`AutoEncoder` in torchgeodemo.models) and patch it.
_ae_cls = _tgd_models2.AutoEncoder
if not getattr(_ae_cls, "_lr_patched", False):
    _ae_cls.configure_optimizers = _patched_configure_optimizers
    _ae_cls._lr_patched = True
    print(f"Patched {_ae_cls.__name__}.configure_optimizers — schedule now driven by _CURRENT_LR_CFG")
else:
    print(f"{_ae_cls.__name__}.configure_optimizers already patched")

print(f"Current LR cfg (will be overridden per-dim in the loop): {_CURRENT_LR_CFG}")

In [ ]:
# --- Selective wipe: force 100D and 128D to retrain under the new LR schedule ---
# The earlier WIPE_STALE cell nuked everything once. After that retrain, only 100/128
# are still under-converged. This cell removes only those outputs (and their
# checkpoints / latent CSVs / model files / training logs) so the training loop
# below will rebuild just those two dims with the new optimizer schedule.

WIPE_HIGH_DIM = True  # set True to force 100/128D to retrain, then leave False

if WIPE_HIGH_DIM:
    import glob, shutil
    dims_to_wipe = [100, 128]
    removed = 0
    for d in dims_to_wipe:
        patterns = [
            f"{output_dir}/models/stability_{d}d_run_*__*",   # model.pth, latent.csv, info.txt
            f"{output_dir}/data/stability_checkpoint_{d}d.pkl",
            f"{output_dir}/models/yamls/config_*_{d}d_*.yaml",
        ]
        for pat in patterns:
            for p in glob.glob(pat):
                if os.path.isdir(p):
                    shutil.rmtree(p); removed += 1
                else:
                    os.remove(p); removed += 1
        # Also wipe the per-dim CSVLogger log dirs
        log_glob = f"{output_dir}/models/logs/log_stability_{d}d_run_*"
        for p in glob.glob(log_glob):
            shutil.rmtree(p); removed += 1
    # And the global stability_results.pkl (will be regenerated from per-dim checkpoints)
    agg = f"{output_dir}/data/all_stability_results.pkl"
    if os.path.exists(agg):
        os.remove(agg); removed += 1
    print(f"Removed {removed} files/dirs for dims {dims_to_wipe}")
    print("Re-run the training cell below to retrain 100D and 128D with the new LR schedule.")
else:
    print("WIPE_HIGH_DIM=False — set True once to retrain 100D and 128D.")

In [ ]:
def load_loss_history(working_dir, latent_dim, run_idx, n_epochs):
    """Load per-epoch reconstruction loss from the CSVLogger metrics.csv.
    Picks the most recent version_N directory for this model."""
    run_name = f"{latent_dim}d_run_{run_idx}"
    # CSVLogger replaces '__' with '_' in the log dir name
    log_name = f"log_stability_{run_name}_ae_{latent_dim}d_{run_name}_{n_epochs}ep_v1"
    log_dir = f"{working_dir}/logs/{log_name}"
    if not os.path.isdir(log_dir):
        return None
    versions = sorted(
        [d for d in os.listdir(log_dir) if d.startswith("version_")],
        key=lambda d: int(d.split("_")[1]),
    )
    if not versions:
        return None
    csv_path = f"{log_dir}/{versions[-1]}/metrics.csv"
    if not os.path.exists(csv_path):
        return None
    df_log = pd.read_csv(csv_path)
    # Keep one row per epoch (the row with recon_loss_epoch populated)
    df_ep = df_log.dropna(subset=["recon_loss_epoch"])[["epoch", "recon_loss_epoch"]]
    df_ep = df_ep.drop_duplicates(subset="epoch").sort_values("epoch")
    return df_ep.reset_index(drop=True)


print("=" * 80)
print(f"Training identity AE for ALL bottleneck sizes: {bottleneck_sizes}")
print(f"Epoch budgets: {n_epochs_by_dim}")
print("=" * 80)

all_ae_embeddings = {}
all_ae_reco_errors = {}
all_ae_loss_history = {}  # {dim: DataFrame(epoch, recon_loss_epoch)}

for latent_dim in bottleneck_sizes:
    n_epochs = n_epochs_by_dim[latent_dim]
    print(f"\n{'='*80}\nBOTTLENECK SIZE: {latent_dim}D  (n_epochs={n_epochs})\n{'='*80}")

    # Configure optimizer schedule for this bottleneck (consumed by patched configure_optimizers)
    _CURRENT_LR_CFG.clear()
    _CURRENT_LR_CFG.update(lr_cfg_by_dim.get(latent_dim, default_lr_cfg))
    print(f"  LR cfg: {_CURRENT_LR_CFG}")

    checkpoint = load_checkpoint(latent_dim)
    if checkpoint is not None and checkpoint['n_runs'] >= n_runs and checkpoint.get('n_epochs') == n_epochs:
        print(f"Found complete checkpoint for {latent_dim}D @ {n_epochs}ep, skipping training")
        all_ae_embeddings[latent_dim] = checkpoint['embeddings_list']
        all_ae_reco_errors[latent_dim] = checkpoint['reco_errors_list']
        all_ae_loss_history[latent_dim] = checkpoint.get('loss_history')
        if all_ae_loss_history[latent_dim] is None:
            all_ae_loss_history[latent_dim] = load_loss_history(
                f"{output_dir}/models", latent_dim, 0, n_epochs
            )
        continue

    ae_embeddings_list = []
    ae_reco_errors_list = []
    ae_seeds = [base_seed + i * 1000 for i in range(n_runs)]

    for run_idx, seed in enumerate(tqdm(ae_seeds, desc=f"Training {latent_dim}D AE")):
        run_name = f"run_{run_idx}"
        print(f"\n  Run {run_idx + 1}/{n_runs} (seed={seed})")

        embeddings = train_ae_via_yaml(
            data_df=df.copy(),
            run_name=f"{latent_dim}d_{run_name}",
            latent_dim=latent_dim,
            working_dir=f"{output_dir}/models",
            seed=seed,
            n_epochs=n_epochs,
            batch_size=batch_size,
            scaling_type=scaling_type
        )
        ae_embeddings_list.append(embeddings)

        model_nickname = f"stability_{latent_dim}d_{run_name}__ae_{latent_dim}d_{latent_dim}d_{run_name}_{n_epochs}ep_v1"
        model_path = f"{output_dir}/models/{model_nickname}__model.pth"

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model = torch.load(model_path, map_location=device, weights_only=False)
        model.eval()

        with torch.no_grad():
            X_tensor = torch.FloatTensor(X_full).to(device)
            latent = model.encode(X_tensor)
            X_reco = model.decoder(latent).cpu().numpy()
            reco_error = np.mean((X_full - X_reco) ** 2, axis=1)

        ae_reco_errors_list.append(reco_error)
        print(f"    Embeddings: {embeddings.shape}, Mean reco error: {reco_error.mean():.6f}")

    loss_hist = load_loss_history(f"{output_dir}/models", latent_dim, 0, n_epochs)
    if loss_hist is not None:
        print(f"    Loaded loss history: {len(loss_hist)} epochs, final loss={loss_hist['recon_loss_epoch'].iloc[-1]:.6f}")
    else:
        print(f"    [warn] No loss history found for {latent_dim}D")

    all_ae_embeddings[latent_dim] = ae_embeddings_list
    all_ae_reco_errors[latent_dim] = ae_reco_errors_list
    all_ae_loss_history[latent_dim] = loss_hist

    # Save checkpoint
    checkpoint_path = get_checkpoint_path(latent_dim)
    with open(checkpoint_path, 'wb') as f:
        pickle.dump({
            'bottleneck_size': latent_dim,
            'embeddings_list': ae_embeddings_list,
            'reco_errors_list': ae_reco_errors_list,
            'loss_history': loss_hist,
            'n_runs': len(ae_embeddings_list),
            'n_epochs': n_epochs,
        }, f)
    print(f"  Checkpoint saved: {checkpoint_path}")

print(f"\n{'='*80}\nCOMPLETED training for all {len(bottleneck_sizes)} bottleneck sizes\n{'='*80}")

Training identity AE for ALL bottleneck sizes: [128, 100, 64, 32, 16, 8, 4, 2]
Epoch budgets: {2: 500, 4: 500, 8: 500, 16: 500, 32: 500, 64: 500, 100: 1500, 128: 1500}

BOTTLENECK SIZE: 128D  (n_epochs=1500)


Training 128D AE:   0%|          | 0/1 [00:00<?, ?it/s]


  Run 1/1 (seed=20210321)
[128d_run_0] Training (seed=20210321, epochs=1500)

geodemo_config={'autoencoder': {'batch_size': 0.01, 'decoder': {'activation': 'Identity', 'sizes': [128, 221, 314]}, 'encoder': {'activation': 'Identity', 'sizes': [314, 221, 128]}, 'max_epochs': 1500, 'nickname': 'ae_128d_128d_run_0_1500ep', 'random_seed': 20210321, 'save_latent': 'csv', 'use_covariance_loss': False, 'version': '1'}, 'data': {'id_col': 'OA', 'nickname': 'stability_128d_run_0', 'source': '../AE_outputs/retraining_stability_500epochs_linscaling_identity/models/temp_data_128d_run_0.parquet'}, 'working_dir': '../AE_outputs/retraining_stability_500epochs_linscaling_identity/models'}


ae_args={'verbose': True, 'encoder_activation': 'Identity', 'use_covariance_loss': False, 'decoder_sizes': [128, 221, 314, 408], 'decoder_activation': 'Identity'}

AutoEncoder(
  (dcc_criterion): MSELoss()
  (encoder): MLP(
    (mlp): Sequential(
      (0): Linear(in_features=408, out_features=314, bias=True)
     

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
You are using a CUDA device ('NVIDIA RTX A500 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name          | Type    | Params | Mode  | FLOPs
----------------------

Epoch 1499: 100%|██████████| 100/100 [00:02<00:00, 47.26it/s, v_num=0_1, recon_loss_step=0.00307, train_loss_step=0.00307, recon_loss_epoch=0.00308, train_loss_epoch=0.00308]

...

    Embeddings: (188880, 100), Mean reco error: 0.000190
    Loaded loss history: 1500 epochs, final loss=0.002877
  Checkpoint saved: ../AE_outputs/retraining_stability_500epochs_linscaling_identity/data/stability_checkpoint_100d.pkl

BOTTLENECK SIZE: 64D  (n_epochs=500)


Training 64D AE:   0%|          | 0/1 [00:00<?, ?it/s]


  Run 1/1 (seed=20210321)
[64d_run_0] Training (seed=20210321, epochs=500)

geodemo_config={'autoencoder': {'batch_size': 0.01, 'decoder': {'activation': 'Identity', 'sizes': [64, 178, 293]}, 'encoder': {'activation': 'Identity', 'sizes': [293, 178, 64]}, 'max_epochs': 500, 'nickname': 'ae_64d_64d_run_0_500ep', 'random_seed': 20210321, 'save_latent': 'csv', 'use_covariance_loss': False, 'version': '1'}, 'data': {'id_col': 'OA', 'nickname': 'stability_64d_run_0', 'source': '../AE_outputs/retraining_stability_500epochs_linscaling_identity/models/temp_data_64d_run_0.parquet'}, 'working_dir': '../AE_outputs/retraining_stability_500epochs_linscaling_identity/models'}


ae_args={'verbose': True, 'encoder_activation': 'Identity', 'use_covariance_loss': False, 'decoder_sizes': [64, 178, 293, 408], 'decoder_activation': 'Identity'}

AutoEncoder(
  (dcc_criterion): MSELoss()
  (encoder): MLP(
    (mlp): Sequential(
      (0): Linear(in_features=408, out_features=293, bias=True)
      (1): Ident

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name          | Type    | Params | Mode  | FLOPs
----------------------------------------------------------
0 | dcc_criterion | MSELoss | 0      | train | 0    
1 | encoder       | MLP     | 183 K  | train | 0    
2 | decoder       | MLP     | 183 K  | train | 0    
----------------------------------------------------------
367 K     Trainable params
0         Non-trainable params
367 K     Total params
1.470     Total estimated model

Epoch 12:  39%|███▉      | 39/100 [00:01<00:01, 31.50it/s, v_num=4_5, recon_loss_step=0.00575, train_loss_step=0.00575, recon_loss_epoch=0.00587, train_loss_epoch=0.00587] 

## AE Reconstruction Error Stability

In [ ]:
print("=" * 80)
print("Computing Identity-AE RMSE for each bottleneck (single run per size)")
print("=" * 80)

stability_results = {}

for latent_dim in bottleneck_sizes:
    ae_reco_errors_list = all_ae_reco_errors[latent_dim]  # list of length n_runs (=1)
    rmse_per_run = [np.sqrt(err.mean()) for err in ae_reco_errors_list]
    rmse_mean = float(np.mean(rmse_per_run))

    stability_results[latent_dim] = {
        'rmse_per_run': rmse_per_run,
        'rmse_mean': rmse_mean,
        'rmse_std': 0.0,  # single run
        'reco_errors': ae_reco_errors_list,
    }
    print(f"  {latent_dim}D: RMSE = {rmse_mean*100:.4f}%")

with open(f"{output_dir}/data/all_stability_results.pkl", 'wb') as f:
    pickle.dump(stability_results, f)
print(f"\nResults saved to {output_dir}/data/all_stability_results.pkl")

summary_df = pd.DataFrame([
    {'Dimension': dim, 'RMSE (%)': stability_results[dim]['rmse_mean'] * 100}
    for dim in bottleneck_sizes
])
print("\n" + "=" * 60)
print("IDENTITY-AE RECONSTRUCTION ERROR")
print("=" * 60)
print(summary_df.to_string(index=False))

In [ ]:
# --- Training loss curves per bottleneck ---
# Sanity check that each model has converged (especially 100/128D which use 1500ep).

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cmap = plt.cm.viridis
dims_sorted = sorted(bottleneck_sizes)
colors = {d: cmap(i / max(1, len(dims_sorted) - 1)) for i, d in enumerate(dims_sorted)}

for d in dims_sorted:
    hist = all_ae_loss_history.get(d)
    if hist is None or len(hist) == 0:
        print(f"[warn] no loss history for {d}D")
        continue
    label = f"{d}D ({n_epochs_by_dim[d]}ep)"
    axes[0].plot(hist['epoch'], hist['recon_loss_epoch'], color=colors[d], linewidth=1.4, label=label)
    axes[1].plot(hist['epoch'], hist['recon_loss_epoch'], color=colors[d], linewidth=1.4, label=label)

axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Reconstruction loss (MSE)')
axes[0].set_title('Linear scale')
axes[0].grid(True, alpha=0.3)
axes[0].legend(frameon=False, fontsize=9, ncol=2)

axes[1].set_yscale('log')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Reconstruction loss (MSE, log)')
axes[1].set_title('Log scale')
axes[1].grid(True, alpha=0.3, which='both')
axes[1].legend(frameon=False, fontsize=9, ncol=2)

fig.suptitle('Identity-AE training loss vs epoch', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{output_dir}/identity_ae_loss_curves.png", dpi=300, bbox_inches='tight')
plt.show()

# --- Convergence diagnostic: relative drop in loss over the last 10% of training ---
print("\nConvergence diagnostic (loss drop over last 10% of training):")
for d in dims_sorted:
    hist = all_ae_loss_history.get(d)
    if hist is None or len(hist) < 20:
        continue
    n = len(hist)
    tail = hist['recon_loss_epoch'].iloc[int(0.9 * n):]
    rel_drop = (tail.iloc[0] - tail.iloc[-1]) / tail.iloc[0] * 100
    print(f"  {d:>3}D: final={hist['recon_loss_epoch'].iloc[-1]:.6f}  "
          f"drop over last 10%: {rel_drop:+.2f}%")

## RMSE vs Dimension with Stability Overlay

Overlay the stability results (mean ± std) from the 10 retraining runs on the RMSE vs dimension plot from notebook 3.

## Comparison: Identity AE vs Nonlinear AE vs PCA

Identity activations collapse the AE to a linear map, so it should track PCA closely. Here
we compare:

- **Identity AE** — single run per bottleneck (deterministic up to optimisation noise).
- **Nonlinear AE (LeakyReLU)** — 10-run mean ± std from the main stability experiment (notebook 2).
- **PCA** — closed-form baseline (cached from notebook 3).

In [ ]:
# --- Load PCA baseline (cached, full-data) ---
pca_cache_path = "../AE_outputs/engcensus_all/pca_rmse_cache.pkl"
with open(pca_cache_path, 'rb') as f:
    pca_rmse = pickle.load(f)
canonical_dims = [2, 4, 8, 16, 32, 64, 100, 128]
pca_by_dim = {d: pca_rmse[i] for i, d in enumerate(canonical_dims) if i < len(pca_rmse)}

# --- Load nonlinear AE 10-run stability results (from notebook 2) ---
nonlinear_path = "../AE_outputs/retraining_stability_500epochs_linscaling/data/all_stability_results.pkl"
with open(nonlinear_path, 'rb') as f:
    nonlinear_results = pickle.load(f)
print(f"Loaded nonlinear-AE stability results from {nonlinear_path}")

# --- Assemble series ---
dims_sorted = sorted(bottleneck_sizes)
id_ae_means   = np.array([stability_results[d]['rmse_mean']  for d in dims_sorted]) * 100
nl_ae_means   = np.array([nonlinear_results[d]['rmse_mean']  for d in dims_sorted]) * 100
nl_ae_stds    = np.array([nonlinear_results[d]['rmse_std']   for d in dims_sorted]) * 100
pca_vals      = np.array([pca_by_dim[d] for d in dims_sorted]) * 100

# --- Plot ---
COL_AE_NL = '#2b83ba'
COL_AE_ID = '#5e3c99'
COL_PCA   = '#d7191c'

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(dims_sorted, pca_vals, 's-', color=COL_PCA,
        label='PCA (closed form)', markersize=7, linewidth=1.8)
ax.plot(dims_sorted, id_ae_means, '^-', color=COL_AE_ID,
        label='Identity AE (1 run)', markersize=7, linewidth=1.8)
ax.errorbar(dims_sorted, nl_ae_means, yerr=nl_ae_stds, fmt='o-', color=COL_AE_NL,
            label='Nonlinear AE (LeakyReLU, 10-run mean ± std)',
            markersize=7, linewidth=1.8, capsize=4)

ax.set_xscale('log', base=2)
ax.set_xticks(dims_sorted); ax.set_xticklabels(dims_sorted)
ax.set_xlabel('Latent Dimension', fontsize=12)
ax.set_ylabel('RMSE (%)', fontsize=12)
ax.set_title('Identity AE vs Nonlinear AE vs PCA', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.legend(frameon=False, fontsize=10, loc='upper right')

plt.tight_layout()
plt.savefig(f"{output_dir}/identity_ae_vs_nonlinear_ae_vs_pca.png", dpi=300, bbox_inches='tight')
plt.savefig(f"{output_dir}/identity_ae_vs_nonlinear_ae_vs_pca.pdf", bbox_inches='tight')
plt.show()

# --- Summary table ---
cmp_df = pd.DataFrame([
    {'Dimension': d,
     'PCA (%)':                 pca_by_dim[d] * 100,
     'Identity AE (%)':         stability_results[d]['rmse_mean'] * 100,
     'Nonlinear AE mean (%)':   nonlinear_results[d]['rmse_mean'] * 100,
     'Nonlinear AE std (%)':    nonlinear_results[d]['rmse_std']  * 100,
     'IdAE - PCA (pp)':        (stability_results[d]['rmse_mean'] - pca_by_dim[d]) * 100,
     'NL AE - PCA (pp)':       (nonlinear_results[d]['rmse_mean'] - pca_by_dim[d]) * 100}
    for d in dims_sorted
])
print("\n" + "=" * 110)
print("Identity AE vs Nonlinear AE vs PCA")
print("=" * 110)
print(cmp_df.to_string(index=False))
cmp_df.to_csv(f"{output_dir}/data/identity_ae_vs_nonlinear_ae_vs_pca.csv", index=False)